In [81]:
import pandas as pd
import numpy as np
import openml
from sklearn.preprocessing import StandardScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [ ]:
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs

task = openml.tasks.get_task(task_ids[0])
dataset = task.get_dataset()
X, y, _, _ = dataset.get_data(target=task.target_name, dataset_format="dataframe")
task_ids

[363612,
 363613,
 363614,
 363615,
 363616,
 363618,
 363619,
 363620,
 363621,
 363623,
 363624,
 363625,
 363626,
 363627,
 363628,
 363629,
 363630,
 363631,
 363632,
 363671,
 363672,
 363673,
 363674,
 363675,
 363676,
 363677,
 363678,
 363679,
 363681,
 363682,
 363683,
 363684,
 363685,
 363686,
 363689,
 363691,
 363693,
 363694,
 363696,
 363697,
 363698,
 363699,
 363700,
 363702,
 363704,
 363705,
 363706,
 363707,
 363708,
 363711,
 363712]

In [62]:
dfs = pd.read_csv("/Users/kingmopser/BachelorThesis/BachelorsThesisCode/task_metadata_tabarena51.csv")

characteristics=dfs.loc[dfs["name"].isin(["QSAR_fish_toxicity","QSAR-TID-11","wine_quality","healthcare_insurance_expenses","Another-Dataset-on-used-Fiat-50","miami_housing"]),:][["tid","name","NumberOfFeatures","target_feature","NumberOfFeatures","NumberOfInstances","NumberOfNumericFeatures","NumberOfSymbolicFeatures"]]
table_df_latex=characteristics.to_latex(caption="Dataset characterstics including shape and dimensions.")

with open("df_table.tex","w") as f:
    f.write(table_df_latex)

In [59]:
table_df_latex

'\\begin{table}\n\\caption{Dataset characterstics including shape and dimensions.}\n\\begin{tabular}{lrrrlrlrr}\n\\toprule\n & NumberOfNumericFeatures & NumberOfSymbolicFeatures & tid & name & NumberOfFeatures & target_feature & NumberOfFeatures & NumberOfInstances \\\\\n\\midrule\n23 & 4.000000 & 3.000000 & 363675 & healthcare_insurance_expenses & 7.000000 & charges & 7.000000 & 1338.000000 \\\\\n33 & 15.000000 & 1.000000 & 363686 & miami_housing & 16.000000 & SALE_PRC & 16.000000 & 13776.000000 \\\\\n39 & 1025.000000 & 0.000000 & 363697 & QSAR-TID-11 & 1025.000000 & MEDIAN_PXC50 & 1025.000000 & 5742.000000 \\\\\n40 & 7.000000 & 0.000000 & 363698 & QSAR_fish_toxicity & 7.000000 & LC50 & 7.000000 & 907.000000 \\\\\n48 & 12.000000 & 1.000000 & 363708 & wine_quality & 13.000000 & median_wine_quality & 13.000000 & 6497.000000 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n'

#### testing



In [63]:
characteristics

,tid,name,NumberOfFeatures,target_feature,NumberOfFeatures,NumberOfInstances,NumberOfNumericFeatures,NumberOfSymbolicFeatures
23,363675,healthcare_insurance_expenses,7.0,charges,7.0,1338.0,4.0,3.0
33,363686,miami_housing,16.0,SALE_PRC,16.0,13776.0,15.0,1.0
39,363697,QSAR-TID-11,1025.0,MEDIAN_PXC50,1025.0,5742.0,1025.0,0.0
40,363698,QSAR_fish_toxicity,7.0,LC50,7.0,907.0,7.0,0.0
48,363708,wine_quality,13.0,median_wine_quality,13.0,6497.0,12.0,1.0


#### loading all datasets


In [ ]:
#def preprocessing(dataset,):
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs
  
characteristics
tables = dict()
for id in characteristics["tid"]:
    task = openml.tasks.get_task(id)
    df = task.get_dataset()
    X, y, _, _ = df.get_data(target=task.target_name, dataset_format="dataframe")
    tables.update({df.name: {"X":X.values,"y":y.values.reshape(-1,1)}})
    

In [191]:
def PreProcessing(name,data,test = 0.2):
    
    X = data.get("X","")
    y = data.get("y","")
    
    is_binary = True if name == "QSAR-TID-11" else False
        
    if is_binary:    
        print("correct detected")
        X = X.loc[:, X.nunique() > 1]
                

    num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    #split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test)
    
        
    preprocessor = ColumnTransformer([("numeric",StandardScaler(),num_cols),
                             ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)])
    
    yScaler = StandardScaler()
    
    X_train= preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)
    
    y_train = yScaler.fit_transform(y_train)
    y_test = yScaler.transform(y_test)
    
    return X_train, X_test, y_train, y_test, yScaler
    
    